<a id="inizio"></a>
# 10 · Agenti per il coding

⏱ ~55 min &nbsp;·&nbsp; Dati: `../Dati/impianti_fv.csv`, `../Dati/letture_pod_2025.csv`

**In questo notebook impariamo a**

- usare Copilot in VS Code per completare, chiedere e far spiegare
- leggere uno script scritto da un agente prima di lanciarlo
- riconoscere type hint, `@dataclass`, decoratori, generatori e `**kwargs`

<a id="indice"></a>
**Indice**

- [1. Copilot in VS Code](#sez-1)
- [2. Glossario](#sez-2)
- [3. Prompt e contesto](#sez-3)
- [4. Le tre regole](#sez-4)
- [5. Leggere il codice scritto da un agente](#sez-5)
    - [5.1 Lo script](#sez-5-1)
    - [5.2 Type hint](#sez-5-2)
    - [5.3 `@dataclass`](#sez-5-3)
    - [5.4 Decoratori](#sez-5-4)
    - [5.5 Generatori](#sez-5-5)
    - [5.6 `**kwargs`](#sez-5-6)
- [6. Esercizi](#sez-6)
    - [Esercizio 10.1 · Il parametro inventato](#es-10-1)
    - [Esercizio 10.2 · Leggere prima di lanciare](#es-10-2)

<a id="sez-1"></a>
## 1. Copilot in VS Code

Copilot lavora in VS Code in due forme. Il completamento propone codice in grigio mentre scriviamo:
**Tab** lo accetta, **Esc** lo rifiuta. La chat scrive codice su richiesta: si apre con **Ctrl+Alt+I**
(su Mac **Ctrl+Cmd+I**), oppure inline in una cella con **Ctrl+I**.

La chat ha tre modalità, dal menu in basso.

| Modalità | Cosa fa |
|---|---|
| Ask | risponde; il codice lo copiamo noi |
| Plan | scrive i passi senza toccare i file |
| Agent | modifica i file mostrando le differenze, lancia comandi chiedendo conferma |

Claude Code e Codex funzionano allo stesso modo, nel terminale o in VS Code.

In [ ]:
# Completamento: scriviamo firma e docstring, Invio, e Copilot propone il corpo in grigio
def prezzo_scontato(prezzo, sconto_percento):
    """Prezzo dopo lo sconto, arrotondato ai centesimi."""
    return round(prezzo * (1 - sconto_percento / 100), 2)


prezzo_scontato(80, 25)  # Output: 60.0

Il corpo si accetta se fa quello che dice la docstring. Il caso noto torna; resta da provare
l'arrotondamento ai centesimi.

In [ ]:
prezzo_scontato(19.99, 10)  # Output: 17.99, non 17.991

<a id="sez-2"></a>
## 2. Glossario

| Parola | Cos'è |
|---|---|
| token | il pezzo di testo che il modello legge e scrive, circa tre quarti di parola; l'uso si misura in token |
| finestra di contesto | quanti token il modello tiene presenti; quando è piena la chat dimentica l'inizio: si apre una chat nuova |
| cache | l'inizio della conversazione tenuto da parte per qualche minuto: rileggerlo costa meno |

| Parola | Cos'è |
|---|---|
| modello | scrive il seguito più probabile, senza verificarlo; nella chat si sceglie dal menu in basso |
| file di istruzioni | regole che l'agente legge a ogni richiesta: `.github/copilot-instructions.md` per Copilot, `CLAUDE.md` per Claude Code, `AGENTS.md` per Codex |

<a id="sez-3"></a>
## 3. Prompt e contesto

Il prompt è la richiesta; il contesto è quello che il modello vede: file aperti, celle selezionate,
testo incollato. Nelle celle qui sotto il commento in testa è il prompt dato alla chat in modalità Ask,
il codice è la risposta.

In [ ]:
import pandas as pd

df = pd.read_csv("../Dati/impianti_fv.csv")
df.info()  # nomi esatti e tipi delle colonne: il contesto da incollare nel prompt

In [ ]:
# Prompt: "Conta gli impianti per anno"
df["anno"].value_counts()

In [ ]:
# Prompt: "Ho un DataFrame df con le colonne id_impianto, comune, provincia, kwp,
#          anno_allaccio, lat, lon. Conta gli impianti per anno_allaccio, in ordine di anno."
per_anno = df["anno_allaccio"].value_counts().sort_index()
per_anno

Il primo prompt non nomina le colonne e la risposta ne usa una che non esiste, `anno`. Per il
secondo controlliamo due cose: gli anni sono in ordine, e i conteggi sommano a 40, le righe di `df`.

In [ ]:
print(per_anno.sum(), len(df))  # Output: 40 40

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Dai alla chat, in modalità Ask, questo prompt:

```text
Ho un DataFrame pandas `df` con queste colonne:
id_impianto (str), comune (str), provincia (str), kwp (float), anno_allaccio (int), lat e lon (float).
Scrivi una funzione `potenza_per_provincia(df)` che restituisce un DataFrame con due colonne,
provincia e kwp, con la somma dei kwp per provincia, ordinato dal più alto al più basso.
Docstring di una riga, niente commenti.
```

Incolla la funzione al posto dei puntini ed esegui. Output atteso: due colonne, `MI` in testa con
323,5 kWp, e la somma della colonna kwp uguale a 737, come in `df`.

</div>

In [ ]:
...


per_provincia = potenza_per_provincia(df)
per_provincia

In [ ]:
# Questa cella dà errore apposta: il traceback si copia intero nella chat
df["kWp"].sum()

In [ ]:
# Prompt: "Spiegami questo errore senza correggerlo: cosa significa, perché succede,
#          dove devo guardare. <traceback intero>"
# Risposta: KeyError, la colonna 'kWp' non esiste. pandas distingue maiuscole e
# minuscole: guarda i nomi esatti delle colonne.
df.columns.tolist()

La spiegazione nomina `KeyError` e la colonna sbagliata, e nella lista c'è il nome giusto, `kwp`.
Solo dopo chiediamo la correzione.

In [ ]:
# Prompt: "Ora correggi la riga"
df["kwp"].sum()

<a id="sez-4"></a>
## 4. Le tre regole

1. **Verifica.** Esegui subito e confronta con un numero noto: la somma dei kwp, il numero di righe.
2. **Chiedi spiegazioni.** Prima "spiegami", poi "correggi", sempre con il traceback intero.
3. **Piccoli passi.** Una richiesta per volta. In Agent le modifiche arrivano come differenze, da
   tenere o annullare blocco per blocco (**Keep** o **Undo**).

<a id="sez-5"></a>
## 5. Leggere il codice scritto da un agente

Uno script scritto da un agente si legge prima di lanciarlo, un blocco alla volta, con quattro domande:

1. La struttura è quella solita: import, costanti, `def`, `main` in fondo?
2. Quali righe fanno il lavoro e quali sono la cornice?
3. Il percorso dei dati esiste, dalla cartella da cui lo lanciamo?
4. Il numero torna?

<a id="sez-5-1"></a>
### 5.1 Lo script

In [ ]:
# Prompt (modalità Agent): "Scrivi uno script che legge le letture, somma i kWh per POD e salva un CSV"
"""Totale dei consumi per POD dal file delle letture mensili."""

import logging
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

logger = logging.getLogger(__name__)

DATA_DIR = Path("../Dati")
OUTPUT = Path("totale_per_pod.csv")

In testa la docstring di modulo, gli import e le costanti, in maiuscolo. `logging`, `dataclasses` e
`pathlib` sono della libreria standard; `logger.info(...)` è un `print` con il livello davanti.

In [ ]:
@dataclass
class Config:
    path: Path
    sep: str = ";"
    decimal: str = ","
    encoding: str = "latin-1"


config = Config(path=DATA_DIR / "letture_pod_2025.csv")  # la riga che sta in main()
config

`@dataclass` è un decoratore: scrive da solo il costruttore `Config(path=...)` e la stampa.
`sep: str = ";"` è un campo con type hint e default: se non lo passiamo, vale `";"`.

In [ ]:
def leggi_letture(config: Config) -> pd.DataFrame:
    """Legge il CSV delle letture con le opzioni della configurazione."""
    if not config.path.exists():
        raise FileNotFoundError(f"File non trovato: {config.path}")
    return pd.read_csv(
        config.path, sep=config.sep, decimal=config.decimal, encoding=config.encoding
    )


def totale_per_pod(letture: pd.DataFrame, fasce: list[str] | None = None) -> pd.DataFrame:
    """Somma dei kWh per POD, eventualmente solo per alcune fasce."""
    if fasce is not None:
        letture = letture[letture["fascia"].isin(fasce)]
    return letture.groupby("pod")["kwh"].sum().reset_index()

Il lavoro lo fanno `read_csv` e il `groupby`; il resto è cornice. `-> pd.DataFrame` è il type hint
di ritorno; `fasce: list[str] | None = None` è un parametro facoltativo.

In [ ]:
totale_per_pod(leggi_letture(config), fasce=["F1"])  # solo la fascia F1

In [ ]:
print(config.path.exists())  # domanda 3. Output: True
totali = totale_per_pod(leggi_letture(config))
print(round(totali["kwh"].sum(), 1), len(totali))  # domanda 4. Output: 134507.7 6

In [ ]:
def main() -> None:
    logging.basicConfig(level=logging.INFO)
    config = Config(path=DATA_DIR / "letture_pod_2025.csv")
    totali = totale_per_pod(leggi_letture(config))
    totali.to_csv(OUTPUT, index=False)
    logger.info(f"Salvati {len(totali)} POD in {OUTPUT}")


# Le ultime due righe dello script:
# if __name__ == "__main__":
#     main()

`logging.basicConfig` accende il log: senza, `logger.info` non stampa. Il blocco
`if __name__ == "__main__":` fa partire `main()` quando il file viene lanciato, non quando viene importato.

<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Prova tu**

Per ogni riga dello script scrivi cos'è, scegliendo tra `"decoratore"`, `"type hint"`,
`"blocco main"` e `"costante"`. Le risposte sono nelle celle qui sopra.

</div>

In [ ]:
cosa_e = {
    "@dataclass": ...,
    "-> pd.DataFrame": ...,
    'if __name__ == "__main__":': ...,
    "DATA_DIR": ...,
}
cosa_e

Riassunto in una pagina: [Leggere il codice: cosa è cosa](../Schede/Scheda_leggere_codice.md).

<a id="sez-5-2"></a>
### 5.2 Type hint

Le forme più frequenti sono `list[str]`, `dict[str, float]`, `X | None` e `-> None`; Python non le
controlla quando esegue.

In [ ]:
def prezzo_medio(prezzi: dict[str, float], escludi: list[str] | None = None) -> float:
    """Prezzo medio dei prodotti, senza quelli in escludi."""
    escludi = escludi or []
    validi = [p for nome, p in prezzi.items() if nome not in escludi]
    return sum(validi) / len(validi)


print(prezzo_medio({"pane": 2.5, "latte": 1.3, "caffè": 4.2}, escludi=["caffè"]))  # Output: 1.9
print(prezzo_medio({"pane": 2, "latte": 1}))  # int al posto di float, nessun errore. Output: 1.5

<a id="sez-5-3"></a>
### 5.3 `@dataclass`

Su una classe fatta di campi, `@dataclass` scrive da solo il costruttore, la stampa e il confronto con `==`.

In [ ]:
from dataclasses import dataclass


@dataclass
class Libro:
    titolo: str
    autore: str
    pagine: int = 0


libro = Libro("Il nome della rosa", "Umberto Eco", pagine=503)
print(libro)
print(libro == Libro("Il nome della rosa", "Umberto Eco", 503))  # Output: True

<a id="sez-5-4"></a>
### 5.4 Decoratori

Una riga `@nome` sopra un `def` avvolge la funzione in un'altra: `@cache` ricorda i risultati già calcolati.

In [ ]:
from functools import cache


@cache  # senza, fibonacci(80) richiederebbe miliardi di chiamate
def fibonacci(n: int) -> int:
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)


fibonacci(80)  # altri decoratori frequenti: @property, @staticmethod, @pytest.fixture

<a id="sez-5-5"></a>
### 5.5 Generatori

Una funzione con `yield` al posto di `return` consegna un valore alla volta, quando un `for` o `list()` lo chiede.

In [ ]:
def a_blocchi(elementi: list, n: int):
    for i in range(0, len(elementi), n):
        yield elementi[i:i + n]


spesa = ["pane", "latte", "uova", "mele", "pasta"]
list(a_blocchi(spesa, 2))

<a id="sez-5-6"></a>
### 5.6 `**kwargs`

`**kwargs` raccoglie in un dizionario i parametri passati per nome che la funzione non elenca
(`*args` in una tupla quelli per posizione); negli script degli agenti li passa così come sono a
un'altra funzione.

In [ ]:
def ordine(piatto: str, **opzioni) -> str:
    return f"{piatto}: {opzioni}"


def leggi_csv(path: str, **opzioni) -> pd.DataFrame:
    return pd.read_csv(path, **opzioni)


print(ordine("pizza", impasto="integrale", extra="olive"))
leggi_csv("../Dati/impianti_fv.csv", usecols=["comune", "kwp"], nrows=3)

<a id="sez-6"></a>
## 6. Esercizi

<a id="es-10-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 10.1 · Il parametro inventato**

`letture_pod_2025.csv` è un CSV all'italiana: punto e virgola, virgola decimale, encoding latin-1.
Il prompt qui sotto suggerisce a Copilot nomi di parametri che `read_csv` non ha.

**Cosa fare.**

1. Dai alla chat questo prompt e incolla la riga che propone al posto di quella commentata:

   ```text
   Leggi il file ../Dati/letture_pod_2025.csv con pandas: separatore punto e virgola,
   virgola come decimale, encoding latin-1. Usa i parametri separator, decimal_separator ed encoding.
   ```
2. Eseguila e leggi l'ultima riga del traceback: quale parametro non esiste?
3. Con `help(pd.read_csv)` trova i nomi giusti per il separatore e per il decimale.
4. Leggi il file in `letture` con i parametri giusti. Output atteso: 216 righe, 5 colonne, `kwh` di tipo `float64`.

</div>

In [ ]:
# 1-2. incolla qui la riga proposta da Copilot ed eseguila
# letture = pd.read_csv("../Dati/letture_pod_2025.csv", ...)

# 3. la documentazione
# help(pd.read_csv)

# 4. la lettura giusta
letture = ...
letture.info()

In [ ]:
from corso import verifica
verifica("10.1")

<a id="es-10-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio 10.2 · Leggere prima di lanciare**

Un collega ha fatto scrivere a un agente lo script del report settimanale. Dice che gira senza
errori, ma il totale gli sembra strano.

```python
"""Report settimanale: kWh totali per POD."""

from dataclasses import dataclass
from pathlib import Path

import pandas as pd


@dataclass
class Config:
    path: Path = Path("Dati") / "letture_pod_2025.csv"
    sep: str = ";"
    encoding: str = "latin-1"


def leggi_letture(config: Config) -> pd.DataFrame:
    """Legge il CSV delle letture."""
    return pd.read_csv(config.path, sep=config.sep, encoding=config.encoding)


def report_kwh(letture: pd.DataFrame) -> pd.DataFrame:
    """kWh totali per POD."""
    return letture.groupby("pod")["kwh"].sum().reset_index()


def main() -> None:
    totale = report_kwh(leggi_letture(Config()))
    totale.to_csv("report_settimanale.csv", index=False)
    print(f"Report salvato: {len(totale)} POD")


if __name__ == "__main__":
    main()
```

**Cosa fare.**

1. Compila `risposte`: in `"lavoro vero"` il nome della funzione che fa il calcolo; in
   `"lanciato in una cella"` cosa succede incollando tutto lo script in una cella del notebook,
   scegliendo tra `"FileNotFoundError"`, `"NameError"` e `"non parte niente"`.
2. Trova il dettaglio che manca nella lettura del file e completa `leggi_letture`, che qui riceve
   direttamente il percorso. Output atteso: sei POD, 134507,7 kWh in tutto.

*Suggerimento:* guarda `totale.dtypes`: se `kwh` non è un numero, il problema è in quello che `read_csv` riceve.

</div>

In [ ]:
risposte = {
    "lavoro vero": ...,
    "lanciato in una cella": ...,
}


def leggi_letture(path: str) -> pd.DataFrame:
    """Legge il CSV delle letture."""
    ...


def report_kwh(letture: pd.DataFrame) -> pd.DataFrame:
    """kWh totali per POD."""
    return letture.groupby("pod")["kwh"].sum().reset_index()


totale = report_kwh(leggi_letture("../Dati/letture_pod_2025.csv"))
totale

In [ ]:
from corso import verifica
verifica("10.2")

---

**Fine del notebook 10.** Precedente: [E3 · Esercitazione 3](Esercitazione_3.ipynb) &nbsp;·&nbsp; Prossimo: [11 · Capstone: il carico del Nord e la temperatura](11_Capstone.ipynb)

<sub>Blocco 4 · Agenti per il coding e caso d'uso end-to-end · Giornata 2 · Aula Avanzata</sub>